# COMBINE ANNOTATIONS

In [ ]:
import json
import os

import numpy as np
from pymongo import MongoClient
from datetime import datetime, timezone
from collections import defaultdict, Counter


import networkx as nx
from scipy.optimize import linear_sum_assignment


# keep pixel conversions only

def ensure_pixels(bbox, norm_w, norm_h):
    # if bbox values look normalized (<=1) convert to pixels
    if max(bbox) <= 1.0:
        return [float(bbox[0]) * norm_w, float(bbox[1]) * norm_h, float(bbox[2]) * norm_w, float(bbox[3]) * norm_h]
    return [float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])]

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    unionArea = (boxA[2]-boxA[0])*(boxA[3]-boxA[1]) + (boxB[2]-boxB[0])*(boxB[3]-boxB[1]) - interArea
    return interArea / unionArea if unionArea > 0 else 0


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h
    area1 = max(0, (box1[2] - box1[0])) * max(0, (box1[3] - box1[1]))
    area2 = max(0, (box2[2] - box2[0])) * max(0, (box2[3] - box2[1]))
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def combine_boxes(boxes, method="mean"):
    """
    boxes: list of [x1, y1, x2, y2]
    """
    boxes = np.array(boxes)

    if method == "mean":
        # Average of all coordinates
        return boxes.mean(axis=0).tolist()

    elif method == "median":
        # More robust to outliers
        return np.median(boxes, axis=0).tolist()

    elif method == "union":
        # Smallest box that contains all boxes
        x1 = boxes[:, 0].min()
        y1 = boxes[:, 1].min()
        x2 = boxes[:, 2].max()
        y2 = boxes[:, 3].max()
        return [x1, y1, x2, y2]

    elif method == "intersection":
        # Largest box contained in all boxes
        x1 = boxes[:, 0].max()
        y1 = boxes[:, 1].max()
        x2 = boxes[:, 2].min()
        y2 = boxes[:, 3].min()
        if x2 < x1 or y2 < y1:
            return None  # no common intersection
        return [x1, y1, x2, y2]


def make_group(group_id, bbox, confidence, label=None):
    return {"groupId": group_id, "bbox": bbox, "confidence": confidence, "label": label}


def make_annotation(video_index, watched, totalwatchtimes, groups, annotation_frame, norm_w, norm_h):
    timestamp = datetime.now(timezone.utc).isoformat()
    return {
        "timestamp": timestamp,
        "videoIndex": video_index,
        "videoFolder": f"videos/clip_{video_index:04d}/",
        "videoWatched": watched,
        "totalWatchTimeMs": totalwatchtimes,
        "numberOfGroups": len(groups),
        "groups": groups,
        "videoInfo": {
            "totalFrames": 50,
            "annotationFrame": annotation_frame,
            "coordinateSystem": "pixel",
            "normalizedWidth": norm_w,
            "normalizedHeight": norm_h
        }
    }

# db connection (reuse existing env vars if present)
MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB = 'video_annotations'
client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
coll = db['finegrained_annotations']

os.makedirs('finegrained_group', exist_ok=True)

# gather video folders
videoFolder = coll.aggregate([
    {"$group": {"_id": "$videoFolder"}},
    {"$sort": {"_id": 1}}
])
video_folders = sorted(set([vid['_id'] for vid in list(videoFolder)]))
#print(video_folders)

root = {"completedAt": datetime.now(timezone.utc).isoformat(), "totalVideos": 0, "annotations": []}

# thresholds
GENERAL_IOU = 0.3
INDIV_IOU = 0.7
iou_threshold = 0.4

jsons_path = 'SEKAI_540_3/jsons_step1/'

fine_total_coincidence, total_coincidence, total_total, fine_total_total = 0, 0, 0, 0


for folder in video_folders:
    ind = int(folder.split('_')[1])
    json_file = f'{jsons_path}/{folder}.json'

    with open(json_file, 'r') as f:
        json_data_s1 = json.load(f)

    for frame in [1, 21, 41]:

        current_frame_info={}
        current_frame_info['frame_id'] = frame+1
        current_frame_info['detections'] = []

        
        # fetch groups for this folder/frame and unwind groups
        pipeline = [
            {"$match": {"videoFolder": folder, "videoInfo.annotationFrame": frame}},
            {"$unwind": {"path": "$groups", "preserveNullAndEmptyArrays": False}},
            {"$project": {
                "_id": 0,
                "annotator_id": 1,
                "videoFolder": 1,
                "annotationFrame": "$videoInfo.annotationFrame",
                "bbox": "$groups.bbox",
                "confidence": "$groups.confidence",
                "label": {"$ifNull": ["$groups.name", {"$ifNull": ["$groups.label", ""]} ]}
            }}
        ]
        docs = list(coll.aggregate(pipeline))
        if not docs:
            # no annotations for this frame
            json_data_s1['frames'].append(current_frame_info)
            root["annotations"].append(make_annotation(ind, 0, 0, [], frame, 1920, 1080))
            continue

        # normalization reference from first doc if available
        vi0 = docs[0].get('videoInfo') if 'videoInfo' in docs[0] else {}
        norm_w = vi0.get('normalizedWidth', 1920) if isinstance(vi0, dict) else 1920
        norm_h = vi0.get('normalizedHeight', 1080) if isinstance(vi0, dict) else 1080

        # collect boxes per annotator (ensure pixels)
        per_annot = defaultdict(list)  # annotator -> list of (bbox_px, confidence, label)
        for d in docs:
            aid = d.get('annotator_id', 'anon')
            lbl = d.get('label') or ''
            bbox_raw = d.get('bbox')
            if not bbox_raw or len(bbox_raw) != 4:
                continue
            bbox_px = ensure_pixels(bbox_raw, norm_w, norm_h)
            conf = d.get('confidence', 1.0)
            per_annot[aid].append((bbox_px, conf, lbl))


        # flatten first annotator's boxes and attempt to match across annotators
        all_annotators = list(per_annot.keys())
        groups = []
        gid = 0
        #print(all_annotators)
        
        groupings = []
        total_keys = []
        
        for l_idx, annotator in enumerate(all_annotators):
            seed_boxes = per_annot[annotator]
            ann_to_groups = defaultdict(list)
    
            for idx, (seed_box, seed_conf, seed_group) in enumerate(seed_boxes):
                ann_to_groups[seed_group].append(seed_box)

            groupings.append(ann_to_groups)
            total_keys.extend(groupings[l_idx].keys())
        
        group_labels = list(set(total_keys))
        if 'individual' in group_labels:
            group_labels.remove('individual')
            group_labels.append('individual')
        
        for label in group_labels:


            coincidence = 0
            total = len(groupings)

            
            for grouping in groupings:
                if label in grouping:
                    coincidence +=1

            total_total+=total
            total_coincidence+=coincidence

            all_boxes = []
            if (coincidence/total) >= 0.5:
                bboxes = []
                if label == 'individual':
                    bboxes = [group['bbox'] for group in groups]
                    
                for ann_idx, grouping in enumerate(groupings):
                    for box in grouping[label]:
                        if label == 'individual':
                            max_iou = 0
                            for g_box in bboxes:
                                max_iou = max(max_iou, iou(g_box, box))
                            #print(max_iou)
                            if max_iou <= 0.95:
                                all_boxes.append((all_annotators[ann_idx], box))
                        else:
                            all_boxes.append((all_annotators[ann_idx], box))
                            
            G = nx.Graph()
            for i in range(len(all_boxes)):
                G.add_node(i)
        
            for i in range(len(all_boxes)):
                maximums = defaultdict(int)
                max_ids  = defaultdict(int)

                
                ann_i, box_i = all_boxes[i]
                area = abs(box_i[0] - box_i[2]) * abs(box_i[1] - box_i[3])
                if area < 100*100:
                    iou_threshold = 0.3
                elif area < 500**500:
                    iou_threshold = 0.5
                else:
                    iou_threshold = 0.7
                        
                for j in range(i + 1, len(all_boxes)):
                    ann_j, box_j = all_boxes[j]
        
                    # ✅ Only connect boxes from DIFFERENT annotators

                    if ann_i != ann_j and iou(box_i, box_j) >= iou_threshold:
                        if iou(box_i, box_j) >=  maximums[ann_j]:
                            maximums[ann_j] = iou(box_i, box_j)
                            max_ids[ann_j] = j

                    for j in max_ids.values():
                        G.add_edge(i, j)
        

            clusters = []
            for component in nx.connected_components(G):
                cluster = [all_boxes[i] for i in component]
                annotators_in_cluster = [ann for ann, _ in cluster]
        
                # Optional: flag if same annotator appears twice in a cluster
                # (can happen via transitive connections — see note below)
                clusters.append({
                    "boxes": cluster,
                    "annotators": annotators_in_cluster,
                    "agreement": len(set(annotators_in_cluster))  # how many unique annotators agree
                })

                #print(cluster)
                fine_total_coincidence += len(set(annotators_in_cluster))
                fine_total_total+=coincidence
                if len(set(annotators_in_cluster))/total >= 0.5:
                    cluster_boxes_only = [box for _, box in cluster]
                    box = combine_boxes(cluster_boxes_only, method="mean")

                    groups.append(make_group(gid, box, 1.0, label))
                    if label == 'individual':
                        gid+=1
            gid+=1

        bboxes = [group['bbox'] for group in groups]
        
        for idx, box in enumerate(bboxes):
            current_frame_info['detections'].append({'track_id': idx+1, 'bbox':box})

        
    #     # append annotation

        
        root["annotations"].append(make_annotation(ind, 0, 0, groups, frame, norm_w, norm_h))
    
        json_data_s1['frames'].append(current_frame_info)

    jsons_path2 = jsons_path.replace('jsons_step1','jsons_step2')
    os.makedirs(jsons_path2, exist_ok=True)
    with open(f"{json_file.replace('jsons_step1','jsons_step2')}", "w") as f:
        json.dump(json_data_s1, f, indent=4)


print('Total agreement:', total_coincidence/total_total)
print('Fine grained, total agreement:', fine_total_coincidence/fine_total_total)

#print(root)
# write merged file
with open('finegrained_group/all_annotations.json', 'w', encoding='utf-8') as f:
    json.dump(root, f, indent=4)


# READ SBU ANNOTATIONS to generate GT


In [3]:
# READ SBU ANNOTATIONS to generate GT
import json
import pickle
from collections import defaultdict

with open('finegrained_group/all_annotations.json', 'r') as f:
    data = json.load(f)

counter = 15

all_results = [{},{},{}]
map_r = {1:0, 21:1, 41:2}


#results = {}


for annotations in data['annotations']:

    ann_frame = annotations['videoInfo']['annotationFrame']
    gt_id = map_r[ann_frame]
    idx = annotations['videoIndex']

    if idx not in all_results[gt_id]:
        all_results[gt_id][idx-1] = {}
    
    
    #print(idx)
    #print(ann_frame)
    groups = []
    map_group_members = defaultdict(list)
    
    for idx_person, group in enumerate(annotations['groups']):

        #print(group)
        #print(idx_person)
        
        x1, y1, x2, y2 = group['bbox']
        group_label = group['groupId']
        map_group_members[group_label+1].append(idx_person+1)
        #print(group_label)
        dc  = 1
        lvl = 1
        
        GT_list = [idx-1, 1, int(x1), int(y1), int(x2), int(y2), group_label+1, dc, lvl]
        str_to_be_added = [str(k) for k in GT_list]
        str_to_be_added = (" ".join(str_to_be_added))
        f = open('./gt_sekai_ours_'+str(ann_frame+1)+'.txt', "a+")
        f.write(str_to_be_added + "\r\n")
        f.close()

    
    groups = list(map_group_members.values())
    #print(groups)
    
    all_results[gt_id][idx-1][str(0)] = groups

#print(all_results[2])
if all_results[0]:
    save_path = f"gt_sekai_2.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(all_results[0], f)


if all_results[1]:
    save_path = f"gt_sekai_22.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(all_results[1], f)


if all_results[2]:
    save_path = f"gt_sekai_42.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(all_results[2], f)

    #results[idx]
    #print('')


In [6]:
import json
from collections import defaultdict

with open('finegrained_group/all_annotations.json', 'r') as f:
    data = json.load(f)

counter = 15
num_fine_grained = 0
num_coarse_groups = 0
freq = defaultdict(int)
for annotations in data['annotations']:
    ann_frame = annotations['videoInfo']['annotationFrame']
    idx = annotations['videoIndex']
    count_groups = defaultdict(int)
    for group in annotations['groups']:
        #print(group)
        if group['label'] != 'individual':
            count_groups[group['label']]+=1
        if group['label'] == 'individual':
            freq[1]+=1
        num_fine_grained+=1

    for count in count_groups:
        freq[count_groups[count]]+=1
    
    num_coarse_groups+=len(count_groups)


print('Number of Finegrained boxes:', num_fine_grained)
print('Number of Coarse boxes:', num_coarse_groups)

print(freq)

Number of Finegrained boxes: 24331
Number of Coarse boxes: 5151
defaultdict(<class 'int'>, {1: 12795, 3: 818, 2: 3705, 4: 218, 6: 30, 5: 76, 7: 14, 24: 1, 9: 4, 12: 2, 8: 3, 13: 1, 11: 1, 10: 1})


# PLOT ANNOTATED DATA

In [1]:

import json
import cv2
from matplotlib.patches import Rectangle
import numpy as np
import os
import matplotlib.pyplot as plt

os.makedirs("finegrained_fig_annotations", exist_ok=True)

with open('finegrained_group/all_annotations.json', 'r') as f:
    data = json.load(f)

#print(data)
# Palette aligned with finegrained UI
palette = [
    '#a9a9a9', '#e6194b', '#f58231', '#ffe119', '#bfef45', '#3cb44b',
    '#911eb4', '#f032e6', '#fabed4', '#9a6324', '#800000', '#ffd8b1',
    '#808000', '#fffac8', '#4363d8', '#f4a261', '#e76f51', '#2a9d8f',
    '#8ac926', '#ffca3a', '#ff595e', '#ff924c', '#c77dff', '#7209b7',
    '#b5179e', '#ff006e', '#d62828', '#f77f00', '#fcbf49', '#6a994e'
]

palette = [
    "#808080",  # Gray
    "#FF0000",  # Red
    "#00FF00",  # Green
    "#0000FF",  # Blue
    "#FFFF00",  # Yellow
    "#00FFFF",  # Cyan
    "#FF00FF",  # Magenta
    "#000000",  # Black
    "#FFFFFF",  # White

    "#000080",  # Maroon
    "#008000",  # Dark Green
    "#800000",  # Navy
    "#008080",  # Olive
    "#800080",  # Purple
    "#808000",  # Teal
    "#C0C0C0",  # Silver

    "#FFA500",  # Orange
    "#FFC0CB",  # Pink
    "#A52A2A",  # Brown
    "#71B33C",  # Medium Sea Green
    "#FF1493",  # Deep Pink
    "#32CD32",  # Lime Green
    "#B469FF",  # Hot Pink
    "#8B4513",  # Dark Slate Blue
]

# stable color map per label within this run
color_map = {}
next_color = 0


for idx, annotations in enumerate(data.get('annotations', [])):
    #print(annotations)
    video_index = annotations.get('videoIndex')
    
    directory_path = annotations.get('videoFolder', '')
    frame = annotations['videoInfo'].get('annotationFrame')
    img_path = os.path.join(directory_path, f"{frame+1:05d}.jpeg")

    image = cv2.imread(img_path)
    if image is None:
        alt = os.path.join(directory_path, f"{frame+1:05d}.jpg")
        image = cv2.imread(alt)
    if image is None:
        print(f"Missing image: {img_path} (and {alt if 'alt' in locals() else 'none'}) - skipping idx={idx}, frame={frame}")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    # Create figure sized so 1 figure point == 1 image pixel at dpi=100
    fig, ax = plt.subplots(figsize=(w/300, h/300), dpi=300)
    ax.imshow(image)

    # Lock axes to pixel coordinates (origin top-left)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)

    # Draw boxes and labels using pixel coordinates
    #color_map = {}
    #next_color = 0

    for group in annotations.get('groups', []):
        bbox = group.get('bbox', [])
        if len(bbox) != 4:
            continue
        x1, y1, x2, y2 = bbox

        # Convert normalized bbox -> pixel if needed
        coord = annotations['videoInfo'].get('coordinateSystem', '')
        if coord == 'normalized' or max(x1, y1, x2, y2) <= 1.0:
            norm_w = annotations['videoInfo'].get('normalizedWidth', 1920)
            norm_h = annotations['videoInfo'].get('normalizedHeight', 1080)
            px1 = int(round(x1 * norm_w))
            px2 = int(round(x2 * norm_w))
            py1 = int(round(y1 * norm_h))
            py2 = int(round(y2 * norm_h))
        else:
            px1, py1, px2, py2 = int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))

        group_label = group.get('label', group.get('groupId', 0))
        label_key = str(group_label)
        #if label_key == 'individual':
        #    color_map[label_key] = palette[next_color % len(palette)]
        #    next_color += 1  
        if label_key not in color_map:
            color_map[label_key] = palette[next_color % len(palette)]
            next_color += 1
            
        color = color_map[label_key]

        rect = Rectangle((px1, py1), px2 - px1, py2 - py1, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        #ax.text(px1, max(0, py1 - 6), str(group_label), color='white', fontsize=6, backgroundcolor='black')

    #print('')
    ax.axis('off')
    plt.savefig(f"finegrained_fig_annotations/finegrained_{idx}.pdf", dpi=300, bbox_inches='tight', pad_inches=0)
    plt.close()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data (approximate values from the image)
categories = np.arange(0, len(frequencies))

print(np.max(categories))
# COCO (3.5)

ours = frequencies#[17, 20, 19, 12, 10, 8, 6, 5, 4, 3, 3, 2, 2, 2, 2]

# PASCAL VOC (1.4)
# pascal = [70, 22, 8, 3, 2, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

# ImageNet (1.7)
# imagenet = [62, 20, 8, 4, 2, 1.5, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

# SUN (9.8)
# sun = [0.5, 1, 3, 5, 7, 9, 9, 8, 7, 6, 5, 4, 3, 2, 2]

# Create the plot
plt.figure(figsize=(10, 6))

# Plot each line
plt.plot(categories, ours, marker='o', label='COCO (3.5)', linewidth=2, markersize=6)
#plt.plot(categories, pascal, marker='o', label='PASCAL VOC (1.4)', linewidth=2, markersize=6)
#plt.plot(categories, imagenet, marker='s', label='ImageNet (1.7)', linewidth=2, markersize=6)
#plt.plot(categories, sun, marker='^', label='SUN (9.8)', linewidth=2, markersize=6)

# Customize the plot
plt.xlabel('Number of Groups', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(categories)
plt.yticks(np.arange(0, 191, 5))
plt.ylim(0, 200)
plt.xlim(0, max_tam+5)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right', fontsize=10)

# Format y-axis as percentages
# ax = plt.gca()
# ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y)}%'))

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data (approximate values from the image)
categories = np.arange(0,len(frequencies))

# COCO (3.5)
ours = frequencies
# Create the plot
plt.figure(figsize=(10, 6))

# Plot bar chart
plt.bar(categories, ours, width=1.0, alpha=0.8, edgecolor='black', linewidth=0.5)

# Customize the plot
plt.xlabel('Number of Groups', fontsize=12)
plt.ylabel('Frequency on Frames', fontsize=12)
plt.title('Aggregated Group frequency', fontsize=12)
plt.xticks(categories)
plt.yticks(np.arange(0, 195, 5))
plt.ylim(0, 195)
plt.xlim(-0.5, 30)
plt.grid(True, alpha=0.3, axis='y')
#plt.legend(loc='upper right', fontsize=10)

# Format y-axis as percentages
ax = plt.gca()
#ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y)}%'))

plt.tight_layout()
plt.savefig('fig_annotations/Aggregated_Group_Frequency.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.array(avg_score)
if data.size == 0:
    print("avg_score empty — skipping aggregated confidence plot")
else:
    total = len(data)

    # ---- Global confidence score (mean) ----
    global_confidence = np.mean(data)

    # ---- Cumulative rates ----
    rates = {
        "1–5": np.sum((data >= 1) & (data <= 5)) / total,
        "2–5": np.sum((data >= 2) & (data <= 5)) / total,
        "3–5": np.sum((data >= 3) & (data <= 5)) / total,
        "4–5": np.sum((data >= 4) & (data <= 5)) / total,
        "5–5": np.sum(data == 5) / total,
    }

    # ---- Plot ----
    plt.figure(figsize=(8, 6))

    plt.hist(
        data,
        bins=20,
        edgecolor='black',
        linewidth=0.5,
        alpha=0.9
    )

    # Global confidence vertical line
    plt.axvline(
        global_confidence,
        linestyle='--',
        linewidth=2,
        label=f'Global Confidence = {global_confidence:.2f}'
    )

    plt.xlabel('Average Confidence', fontsize=11)
    plt.ylabel('Frequency on Frames', fontsize=11)
    plt.title('Aggregated Confidence Score', fontsize=12)

    plt.xlim(0, 5)
    plt.ylim(0, 255)
    plt.grid(True, alpha=0.3, axis='y')

    # ---- Annotation text ----
    text = (
        f"Global Confidence: {global_confidence:.2f}\n\n" +
        "\n".join([f"Rate {k}: {v*100:.1f}%" for k, v in rates.items()])
    )

    plt.text(
        0.98, 0.95, text,
        transform=plt.gca().transAxes,
        fontsize=10,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', alpha=0.15),
    )

    plt.legend()
    plt.tight_layout()
    plt.savefig('fig_annotations/Aggregated_Confidence_Score.pdf', format='pdf', bbox_inches='tight')
    plt.show()


# LAST FIGURE

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from matplotlib.lines import Line2D

# Load image
image = cv2.imread('videos/clip_0001/00001.jpeg')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(image)

# Left column: numbers
confidence_values = [1, 2, 3, 4, 5]
left_labels = [f"{i+1}." for i in range(len(confidence_values))]

# Right column: your activity names
right_labels = ["Preparing food", "Socializing", "Serving customers", "Browsing food", "Walking", "Shopping", "Selling food", "Eating"]

# Combine into rows:  "1.  Walking", etc.
combined_labels = [f"{left:<3}  {right}" for left, right in zip(left_labels, right_labels)]

# Invisible handles
empty_handles = [Line2D([], [], linestyle="none") for _ in combined_labels]

legend = ax.legend(
    empty_handles,
    combined_labels,
    title="Activities List",
    loc="upper right",
    frameon=False,
    handlelength=0,
    handletextpad=0.2,
    prop={'size': 7.5, 'weight': 'bold'},
    title_fontsize=8
)

# Style text
for text in legend.get_texts():
    text.set_color("white")

legend.get_title().set_color("white")
legend.get_title().set_fontweight("bold")

ax.axis("off")
plt.savefig("fig_annotations/sample_activities_list.pdf", bbox_inches='tight', pad_inches=0)
plt.show()
plt.close()